[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-org/pypath/blob/main/notebooks/module7/02-pytorch-basics.ipynb)

# PyTorch Basics
**Module 7 — Lesson 2 | Estimated time: 35 minutes**

> 💡 Enable GPU: Runtime → Change runtime type → GPU

## Learning Objectives
By the end of this notebook you will be able to:
- Create and manipulate PyTorch tensors and understand broadcasting
- Use `requires_grad`, call `.backward()`, and inspect `.grad`
- Build custom modules using `nn.Module`
- Compose layers with `nn.Sequential`
- Apply common loss functions and optimisers
- Write a complete training loop with `train()` / `eval()` modes
- Load data efficiently using `DataLoader` and `TensorDataset`
- Plot training and validation loss curves

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch:', torch.__version__)
print('Device: ', device)

## 1. Tensors — Creation and Basic Operations

Tensors are PyTorch's fundamental data structure, similar to NumPy arrays but GPU-aware.

In [ ]:
# --- Creation ---
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.zeros(3, 4)
c = torch.ones(2, 3)
d = torch.rand(3, 3)       # uniform [0, 1)
e = torch.randn(3, 3)      # standard normal
f = torch.arange(0, 10, 2, dtype=torch.float32)

print('tensor:  ', a)
print('zeros:   ', b.shape)
print('rand:    ', d)
print('arange:  ', f)

# --- Operations ---
x = torch.randn(3, 3)
y = torch.randn(3, 3)

print('\nElement-wise add:\n', (x + y).round(3))
print('Matrix multiply:\n', (x @ y).round(3))
print('Transpose:\n', x.T.round(3))
print('Reduction (sum):', x.sum().item())
print('Reshape:', x.reshape(1, 9).shape)
print('Squeeze/Unsqueeze:', torch.unsqueeze(a, 0).shape)

# --- Broadcasting ---
v = torch.tensor([[1.], [2.], [3.]])  # (3, 1)
w = torch.tensor([10., 20., 30.])    # (3,)
print('\nBroadcasting (3,1) + (3,):\n', (v + w))

## 2. Moving Tensors to GPU

Use `.to(device)` to move tensors and models between CPU and GPU seamlessly.

In [ ]:
t_cpu = torch.randn(1000, 1000)
t_gpu = t_cpu.to(device)
print('CPU tensor device:', t_cpu.device)
print('GPU tensor device:', t_gpu.device)

# Matrix multiplication on the chosen device
import time
t1 = torch.randn(512, 512).to(device)
t2 = torch.randn(512, 512).to(device)
# warm-up
_ = t1 @ t2
start = time.time()
for _ in range(100):
    result = t1 @ t2
if device == 'cuda': torch.cuda.synchronize()
elapsed = time.time() - start
print(f'\n100× (512×512) matmul on {device}: {elapsed*1000:.1f} ms')

## 3. Autograd — Automatic Differentiation

PyTorch tracks operations on tensors with `requires_grad=True` and builds a **computational graph**. Calling `.backward()` populates `.grad` for every leaf tensor.

In [ ]:
# Simple scalar example
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2 + 2 * x + 1   # y = (x+1)^2
y.backward()
print(f'dy/dx at x=3: {x.grad.item()}  (expected: 2*(3+1) = 8)')

# Vector example
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
z = (x ** 2).sum()  # scalar loss
z.backward()
print('Gradient of sum(x^2):', x.grad)  # 2*x

# Detach from graph when you do NOT want gradients
x_data = x.detach().numpy()  # safe for NumPy interop
print('Detached numpy:', x_data)

# Gradient accumulation demo (zero_grad is important!)
w = torch.tensor(1.0, requires_grad=True)
for step in range(3):
    loss = (w * 2) ** 2
    loss.backward()
    print(f'Step {step+1} grad (without zero_grad): {w.grad.item()}')
# Gradients accumulate — always call optimiser.zero_grad() in training loops

## 4. Building Models with nn.Module

`nn.Module` is the base class for all neural network models. You override `__init__` to define layers and `forward` to define the computation.

In [ ]:
class TwoLayerNet(nn.Module):
    """Custom 2-layer fully-connected network."""
    def __init__(self, in_features, hidden, out_features, dropout=0.3):
        super().__init__()
        self.fc1     = nn.Linear(in_features, hidden)
        self.act     = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.fc2     = nn.Linear(hidden, out_features)
        self.out_act = nn.Sigmoid()

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return self.out_act(x)

net = TwoLayerNet(in_features=10, hidden=32, out_features=1)
print(net)
print('\nParameters:')
for name, param in net.named_parameters():
    print(f'  {name:20s} {tuple(param.shape)}')
total = sum(p.numel() for p in net.parameters())
print(f'\nTotal parameters: {total:,}')

# nn.Sequential alternative
seq_net = nn.Sequential(
    nn.Linear(10, 32),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(32, 1),
    nn.Sigmoid()
)
print('\nSequential net:', seq_net)

## 5. Loss Functions and Optimisers

PyTorch provides standard loss functions in `torch.nn` and optimisers in `torch.optim`.

In [ ]:
# Loss functions
y_pred = torch.sigmoid(torch.randn(8, 1))
y_true_bin = torch.randint(0, 2, (8, 1)).float()
y_logits = torch.randn(8, 5)  # for multi-class
y_class  = torch.randint(0, 5, (8,))

bce_loss = nn.BCELoss()(y_pred, y_true_bin)
mse_loss = nn.MSELoss()(y_pred, y_true_bin)
ce_loss  = nn.CrossEntropyLoss()(y_logits, y_class)

print(f'BCE:             {bce_loss.item():.4f}')
print(f'MSE:             {mse_loss.item():.4f}')
print(f'Cross-Entropy:   {ce_loss.item():.4f}')

# Optimisers
model_demo = nn.Linear(10, 1)
sgd   = torch.optim.SGD(model_demo.parameters(), lr=0.01, momentum=0.9)
adam  = torch.optim.Adam(model_demo.parameters(), lr=1e-3, weight_decay=1e-4)
adamw = torch.optim.AdamW(model_demo.parameters(), lr=1e-3, weight_decay=0.01)
print('\nOptimisers created: SGD, Adam, AdamW')

## 6. Complete Training Loop

A canonical PyTorch training loop:
1. Zero gradients
2. Forward pass
3. Compute loss
4. Backward pass
5. Update parameters
6. Track metrics

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

# Dataset
X_np, y_np = make_classification(n_samples=1000, n_features=10, random_state=42)
X_tr, X_val, y_tr, y_val = train_test_split(X_np, y_np, test_size=0.2, random_state=42)

X_tr_t  = torch.tensor(X_tr,  dtype=torch.float32).to(device)
y_tr_t  = torch.tensor(y_tr,  dtype=torch.float32).unsqueeze(1).to(device)
X_val_t = torch.tensor(X_val, dtype=torch.float32).to(device)
y_val_t = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1).to(device)

# DataLoaders
train_ds = TensorDataset(X_tr_t, y_tr_t)
val_ds   = TensorDataset(X_val_t, y_val_t)
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
val_dl   = DataLoader(val_ds,   batch_size=64)

# Model
model = TwoLayerNet(10, 64, 1, dropout=0.2).to(device)
optimiser = torch.optim.AdamW(model.parameters(), lr=1e-3)
criterion = nn.BCELoss()
scheduler = torch.optim.lr_scheduler.StepLR(optimiser, step_size=20, gamma=0.5)

train_losses, val_losses = [], []

for epoch in range(60):
    # --- Train ---
    model.train()
    epoch_loss = 0.0
    for xb, yb in train_dl:
        optimiser.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimiser.step()
        epoch_loss += loss.item() * len(xb)
    train_losses.append(epoch_loss / len(train_ds))

    # --- Validate ---
    model.eval()
    with torch.no_grad():
        val_preds = model(X_val_t)
        val_loss  = criterion(val_preds, y_val_t).item()
    val_losses.append(val_loss)
    scheduler.step()

print(f'Final train loss: {train_losses[-1]:.4f}')
print(f'Final val   loss: {val_losses[-1]:.4f}')

acc = ((model(X_val_t) > 0.5).float() == y_val_t).float().mean()
print(f'Val accuracy:     {acc.item():.3f}')

## 7. Plotting Loss Curves

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(train_losses, label='Train loss', color='steelblue')
plt.plot(val_losses,   label='Val loss',   color='darkorange', linestyle='--')
plt.xlabel('Epoch')
plt.ylabel('BCE Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.tight_layout()
plt.show()

## Practice Exercises

**Exercise 1 — Custom Module**
Create an `nn.Module` called `MLP` that accepts a list of layer sizes (e.g. `[10, 64, 32, 1]`) and builds a fully-connected network programmatically using `nn.ModuleList`. Include batch normalisation between layers.

**Exercise 2 — Learning Rate Scheduler Comparison**
Train the same model three times using `StepLR`, `CosineAnnealingLR`, and `ReduceLROnPlateau`. Plot the three validation loss curves on a single figure and identify which scheduler converges fastest.

**Exercise 3 — Weight Initialisation**
Experiment with three weight initialisations for `nn.Linear` weights: default (Kaiming uniform), Xavier normal (`torch.nn.init.xavier_normal_`), and all-zeros. Train each for 30 epochs and compare loss curves. Explain why zero initialisation fails.